# Transformer Training

Create a decoder-only Transformer with the package model builder, load labelled feature CSVs through the package dataset loader, train, save a checkpoint, reload it, and run one inference example.

In [ ]:
from datetime import datetime
from pathlib import Path
import random
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PACKAGE_SRC = PROJECT_ROOT / "packages" / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

PROJECT_ROOT

In [ ]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from tradingbot.models import TransformerWindowDataset, build_transformer

DATA_DIR = PROJECT_ROOT / "data" / "historic_candle_feature"
MODEL_DIR = PROJECT_ROOT / "data" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FILE_GLOB = "*_features_triple_barrier.csv"
TIMEFRAME_FILE_PATTERN = "*_15minute_*_features_triple_barrier.csv"  # set None to use every labelled file
CHECKPOINT_PREFIX = "triple_barrier_transformer"
SEED = 42

OUTPUT_TYPE = "classification"  # "classification" or "regression"
CONTEXT_LENGTH = 128
TARGET_OFFSET = 0  # triple-barrier labels are already aligned to the candle being predicted
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
MAX_ROWS_PER_FILE = None  # set None for full training

MODEL_DIM = 24
NUM_HEADS = 4
NUM_LAYERS = 4
DROPOUT = 0.1

LABEL_COLUMNS = [
    "triple_barrier_label_lower",
    "triple_barrier_label_neutral",
    "triple_barrier_label_upper",
]
REGRESSION_COLUMNS = ["triple_barrier_return"]

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

DEVICE

In [ ]:
csv_files = sorted(DATA_DIR.glob(FILE_GLOB))
if TIMEFRAME_FILE_PATTERN is not None:
    csv_files = sorted(path for path in csv_files if path.match(TIMEFRAME_FILE_PATTERN))
if not csv_files:
    pattern = TIMEFRAME_FILE_PATTERN or FILE_GLOB
    raise FileNotFoundError(f"No labelled feature files found at {DATA_DIR} matching {pattern}")

schema = pd.read_csv(csv_files[0], nrows=0).columns.tolist()
excluded_columns = {"timestamp", *LABEL_COLUMNS, *REGRESSION_COLUMNS}
available_feature_columns = [column for column in schema if column not in excluded_columns]

if OUTPUT_TYPE == "classification":
    missing = [column for column in LABEL_COLUMNS if column not in schema]
    target_mode = "class_index"
    target_kwargs = {"label_columns": LABEL_COLUMNS}
    output_dim = len(LABEL_COLUMNS)
else:
    missing = [column for column in REGRESSION_COLUMNS if column not in schema]
    target_mode = "regression"
    target_kwargs = {"target_columns": REGRESSION_COLUMNS}
    output_dim = len(REGRESSION_COLUMNS)

if missing:
    raise ValueError(f"Missing target columns in {csv_files[0]}: {missing}")

matched_files = [path.name for path in csv_files]
print(f"Matched {len(matched_files)} files using pattern: {TIMEFRAME_FILE_PATTERN or FILE_GLOB}")
pd.DataFrame({"matched_file": matched_files})


## Feature Columns

Edit `SELECTED_FEATURE_COLUMNS` to train with a subset. Leave it as `None` to use every available feature column discovered from the labelled CSV schema.

In [ ]:
SELECTED_FEATURE_COLUMNS = None

# Example subset:

SELECTED_FEATURE_COLUMNS = [
  'upper_wick_to_candle_ratio',
  'lower_wick_to_candle_ratio',
  'body_to_candle_ratio',
  'candle_type_standard',
  'candle_type_doji',
  'candle_type_hammer',
  'candle_type_inverted_hammer',
  'candle_type_spinning_top',
  'candle_type_marubozu',
  'candle_color_green',
  'candle_color_red',
  'log_volume_zscore',
  'rsi_7',
  'rsi_14',
  'stochastic_7',
  'stochastic_14',
  'stochastic_rsi_7',
  'stochastic_rsi_14',
  'macd',
  'atr',
  'adx'
  ]

feature_columns = available_feature_columns if SELECTED_FEATURE_COLUMNS is None else list(SELECTED_FEATURE_COLUMNS)
unknown_features = [column for column in feature_columns if column not in available_feature_columns]
if unknown_features:
    raise ValueError(f"Selected features are not present in the dataset: {unknown_features}")
if not feature_columns:
    raise ValueError("Select at least one feature column.")

shuffled_files = csv_files.copy()
shuffled_files = shuffled_files[:1]
random.shuffle(shuffled_files)
split_index = max(1, int(len(shuffled_files) * 0.8))
train_files = shuffled_files[:split_index]
val_files = shuffled_files[split_index:] or shuffled_files[:1]

{
    "files": len(shuffled_files),
    "train_files": len(train_files),
    "val_files": len(val_files),
    "feature_dim": len(feature_columns),
    "feature_columns": feature_columns,
    "target_mode": target_mode,
    "file_pattern": TIMEFRAME_FILE_PATTERN or FILE_GLOB,
    "matched_files": [path.name for path in shuffled_files],
}

In [ ]:
train_dataset = TransformerWindowDataset(
    train_files,
    feature_columns=feature_columns,
    context_length=CONTEXT_LENGTH,
    target_mode=target_mode,
    target_offset=TARGET_OFFSET,
    max_rows_per_file=MAX_ROWS_PER_FILE,
    **target_kwargs,
)
val_dataset = TransformerWindowDataset(
    val_files,
    feature_columns=feature_columns,
    context_length=CONTEXT_LENGTH,
    target_mode=target_mode,
    target_offset=TARGET_OFFSET,
    max_rows_per_file=MAX_ROWS_PER_FILE,
    **target_kwargs,
)

if len(train_dataset) == 0:
    raise ValueError(f"Training dataset is empty. Skipped files: {train_dataset.skipped[:5]}")
if len(val_dataset) == 0:
    raise ValueError(f"Validation dataset is empty. Skipped files: {val_dataset.skipped[:5]}")

pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)

{
    "train_windows": len(train_dataset),
    "val_windows": len(val_dataset),
    "train_skipped": train_dataset.skipped[:5],
    "val_skipped": val_dataset.skipped[:5],
    "label_counts": None if train_dataset.label_counts is None else train_dataset.label_counts.tolist(),
}

In [ ]:
model_config = {
    "input_dim": len(feature_columns),
    "model_dim": MODEL_DIM,
    "num_heads": NUM_HEADS,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "output_dim": output_dim,
    "output_type": OUTPUT_TYPE,
    "feature_columns": feature_columns,
    "file_glob": FILE_GLOB,
    "timeframe_file_pattern": TIMEFRAME_FILE_PATTERN,
    "matched_files": matched_files,
}
builder_config = {
    key: value
    for key, value in model_config.items()
    if key not in {"feature_columns", "file_glob", "timeframe_file_pattern", "matched_files"}
}
model = build_transformer(**builder_config).to(DEVICE)

if OUTPUT_TYPE == "classification":
    class_weight = None
    if train_dataset.label_counts is not None and (train_dataset.label_counts > 0).all():
        counts = torch.tensor(train_dataset.label_counts, dtype=torch.float32, device=DEVICE)
        class_weight = counts.sum() / (counts.numel() * counts)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
else:
    criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

sum(parameter.numel() for parameter in model.parameters())

In [ ]:
def move_batch(batch, device):
    x, y = batch
    return x.to(device), y.to(device)


def classification_metrics(correct, total, true_positive, false_positive, false_negative):
    precision = true_positive / (true_positive + false_positive).clamp_min(1)
    recall = true_positive / (true_positive + false_negative).clamp_min(1)
    f1 = 2 * precision * recall / (precision + recall).clamp_min(1e-12)
    return {
        "accuracy": correct / max(total, 1),
        "precision_macro": precision.mean().item(),
        "recall_macro": recall.mean().item(),
        "f1_macro": f1.mean().item(),
        "precision_by_class": precision.cpu().tolist(),
        "recall_by_class": recall.cpu().tolist(),
        "f1_by_class": f1.cpu().tolist(),
    }


def checkpoint_path_for(objective_type, final_val_accuracy):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    accuracy_text = "na" if final_val_accuracy is None else f"{final_val_accuracy:.4f}"
    return MODEL_DIR / f"{CHECKPOINT_PREFIX}_{objective_type}_{timestamp}_valacc_{accuracy_text}.pt"


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_items = 0

    for batch in loader:
        x, y = move_batch(batch, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(x)
        loss = criterion(prediction, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size

    return total_loss / max(total_items, 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_items = 0

    if OUTPUT_TYPE == "classification":
        class_count = output_dim
        correct = 0
        true_positive = torch.zeros(class_count, dtype=torch.float32, device=DEVICE)
        false_positive = torch.zeros(class_count, dtype=torch.float32, device=DEVICE)
        false_negative = torch.zeros(class_count, dtype=torch.float32, device=DEVICE)

    for batch in loader:
        x, y = move_batch(batch, DEVICE)
        prediction = model(x)
        loss = criterion(prediction, y)

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size

        if OUTPUT_TYPE == "classification":
            predicted = prediction.argmax(dim=-1)
            correct += (predicted == y).sum().item()
            for class_index in range(class_count):
                predicted_class = predicted == class_index
                actual_class = y == class_index
                true_positive[class_index] += (predicted_class & actual_class).sum()
                false_positive[class_index] += (predicted_class & ~actual_class).sum()
                false_negative[class_index] += (~predicted_class & actual_class).sum()

    metrics = {"loss": total_loss / max(total_items, 1)}
    if OUTPUT_TYPE == "classification":
        metrics.update(
            classification_metrics(
                correct,
                total_items,
                true_positive,
                false_positive,
                false_negative,
            )
        )
    return metrics


def prefixed_metrics(prefix, metrics):
    return {f"{prefix}_{key}": value for key, value in metrics.items()}


history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    train_metrics = evaluate(model, train_loader)
    val_metrics = evaluate(model, val_loader)
    row = {
        "epoch": epoch,
        "train_optimized_loss": train_loss,
        **prefixed_metrics("train", train_metrics),
        **prefixed_metrics("val", val_metrics),
    }
    history.append(row)
    print(row)

final_metrics = history[-1]
final_val_accuracy = final_metrics.get("val_accuracy")
saved_checkpoint_path = checkpoint_path_for(OUTPUT_TYPE, final_val_accuracy)
checkpoint_context = {
    "model_state_dict": model.state_dict(),
    "model_config": model_config,
    "feature_columns": feature_columns,
    "file_glob": FILE_GLOB,
    "timeframe_file_pattern": TIMEFRAME_FILE_PATTERN,
    "matched_files": matched_files,
    "train_files": [path.name for path in train_files],
    "val_files": [path.name for path in val_files],
    "label_columns": LABEL_COLUMNS if OUTPUT_TYPE == "classification" else None,
    "target_columns": REGRESSION_COLUMNS if OUTPUT_TYPE == "regression" else None,
    "context_length": CONTEXT_LENGTH,
    "target_offset": TARGET_OFFSET,
    "history": history,
    "final_metrics": final_metrics,
    "objective_type": OUTPUT_TYPE,
    "saved_checkpoint_name": saved_checkpoint_path.name,
}
torch.save(checkpoint_context, saved_checkpoint_path)
print(f"Saved model to {saved_checkpoint_path}")

pd.DataFrame(history)


In [ ]:
checkpoint_path = saved_checkpoint_path if "saved_checkpoint_path" in globals() else sorted(
    MODEL_DIR.glob(f"{CHECKPOINT_PREFIX}_{OUTPUT_TYPE}_*_valacc_*.pt")
)[-1]
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
checkpoint_model_config = checkpoint["model_config"]
checkpoint_builder_config = {
    key: value
    for key, value in checkpoint_model_config.items()
    if key not in {"feature_columns", "file_glob", "timeframe_file_pattern", "matched_files"}
}
inference_model = build_transformer(**checkpoint_builder_config).to(DEVICE)
inference_model.load_state_dict(checkpoint["model_state_dict"])
inference_model.eval()

x, y = val_dataset[0]
with torch.no_grad():
    output = inference_model(x.unsqueeze(0).to(DEVICE))

if checkpoint_model_config["output_type"] == "classification":
    probabilities = torch.softmax(output, dim=-1).squeeze(0).cpu()
    predicted_index = int(probabilities.argmax().item())
    inference_result = {
        "checkpoint": checkpoint_path.name,
        "features_used": checkpoint_model_config["feature_columns"],
        "timeframe_file_pattern": checkpoint_model_config.get("timeframe_file_pattern"),
        "predicted_label": checkpoint["label_columns"][predicted_index],
        "probabilities": dict(zip(checkpoint["label_columns"], probabilities.tolist())),
        "actual_index": int(y.item()),
        "actual_label": checkpoint["label_columns"][int(y.item())],
    }
else:
    prediction = output.squeeze(0).cpu().tolist()
    actual = y.cpu().tolist()
    inference_result = {
        "checkpoint": checkpoint_path.name,
        "features_used": checkpoint_model_config["feature_columns"],
        "timeframe_file_pattern": checkpoint_model_config.get("timeframe_file_pattern"),
        "prediction": dict(zip(checkpoint["target_columns"], prediction)),
        "actual": dict(zip(checkpoint["target_columns"], actual)),
    }

inference_result
